# 🌿 KrishiOS AI — Tomato Leaf Disease Classification (Phase 1)
### Deep Learning Training Pipeline on Kaggle GPU using MobileNetV3-Large & PlantVillage

**Project:** KrishiOS AI — Intelligent Crop Disease Diagnostics System  
**Author:** KrishiOS AI Engineering Team  
**Task:** 10-Class Tomato Leaf Disease & Health State Classification  
**Target Architecture:** MobileNetV3-Large (Pretrained ImageNet transfer learning)  
**Hardware Target:** Kaggle GPU (NVIDIA Tesla T4 / P100) or CPU Fallback  
**Reproducibility:** Seed 42, Stratified 80/10/10 Split, Zero Data Leakage

---
### 📌 Workflow Overview
1. **Environment & Hardware Inspection:** Check GPU availability and CUDA acceleration.
2. **Dynamic Dataset Discovery:** Automatically find PlantVillage files in `/kaggle/input/`.
3. **Data Integrity & Deduplication:** Audit all 10 Tomato classes, filter corrupt files, eliminate duplicate leakage.
4. **Data Splitting & Augmentation:** Stratified 80% Train, 10% Validation, 10% Test.
5. **Model Setup:** MobileNetV3-Large with custom 10-class linear classifier head.
6. **Training Engine:** AdamW optimizer, Cosine Annealing, CrossEntropyLoss, and Early Stopping.
7. **Evaluation:** Independent evaluation on untouched Test set (Accuracy, Macro-F1, Confusion Matrix, Top-3).
8. **Artifact Packaging:** Save weights, mappings, inference script, and ZIP package for KrishiOS backend integration.

In [ ]:
import os
import sys
import time
import json
import random
import shutil
import hashlib
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

# Global reproducibility seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 65)
print("🌾 KrishiOS AI — Environment & Hardware Configuration")
print("=" * 65)
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"Torchvision       : {torchvision.__version__}")
print(f"Compute Device    : {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"GPU Accelerator   : {gpu_name} ({vram_gb:.2f} GB VRAM)")
    print("🚀 Hardware acceleration enabled. Ready for high-throughput training.")
else:
    print("⚠️ No GPU detected. Training will run on CPU (will be slower).")
    print("👉 If running on Kaggle: Click 'Settings' in the right sidebar -> Accelerator -> Select 'GPU T4 x2' or 'GPU P100'.")
print("=" * 65)

## 1. Dataset Discovery & Path Resolution
The pipeline automatically inspects `/kaggle/input/` for the PlantVillage dataset across different common folder structures (`color/`, nested folders, or direct class folders). If running locally, it also checks current directory fallbacks.

In [ ]:
def discover_plantvillage_tomato_root():
    search_roots = [
        Path("/kaggle/input"),
        Path("./plantvillage dataset"),
        Path("../plantvillage dataset"),
        Path("."),
    ]
    target_indicator = "Tomato___Early_blight"
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob(target_indicator):
            if p.is_dir():
                candidate = p.parent
                tomato_dirs = [d for d in candidate.iterdir() if d.is_dir() and d.name.startswith("Tomato___")]
                if len(tomato_dirs) >= 8:
                    return candidate
    return None

DATASET_ROOT = discover_plantvillage_tomato_root()

if DATASET_ROOT is not None:
    print("✅ PlantVillage dataset successfully discovered at:")
    print(f"   {DATASET_ROOT.resolve()}")
else:
    print("❌ ERROR: PlantVillage Tomato dataset could not be found.")
    print("=" * 65)
    print("👉 HOW TO FIX ON KAGGLE:")
    print("   1. Click '+ Add Input' on the top-right Kaggle panel.")
    print("   2. Search for 'PlantVillage' or 'Plant Disease'.")
    print("   3. Add the dataset (e.g., 'abdallahalidev/plantvillage-dataset' or 'emmarex/plantdisease').")
    print("   4. Re-run this cell.")
    print("=" * 65)
    raise FileNotFoundError("PlantVillage dataset directory not located.")

In [ ]:
all_subdirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir() and d.name.startswith("Tomato___")], key=lambda x: x.name)

print("=" * 65)
print(f"🔍 Discovered {len(all_subdirs)} Tomato Class Subdirectories:")
print("=" * 65)

dataset_records = []
corrupt_files = []
file_hashes = defaultdict(list)

for class_dir in all_subdirs:
    c_name = class_dir.name
    files = list(class_dir.glob("*.*"))
    valid_count = 0
    for f in files:
        if f.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue
        try:
            with Image.open(f) as img:
                w, h = img.size
                if w < 10 or h < 10:
                    corrupt_files.append((str(f), "Invalid dimensions"))
                    continue
            with open(f, "rb") as fp:
                file_hash = hashlib.md5(fp.read()).hexdigest()
            file_hashes[file_hash].append(f)
            dataset_records.append({
                "filepath": f,
                "filename": f.name,
                "class_name": c_name,
                "hash": file_hash
            })
            valid_count += 1
        except Exception as e:
            corrupt_files.append((str(f), str(e)))
    print(f"  • {c_name:<45} : {valid_count:>5} images")

print("=" * 65)
print(f"Total Valid Tomato Images Found : {len(dataset_records)}")
print(f"Total Corrupt / Broken Files    : {len(corrupt_files)}")

dup_groups = [flist for flist in file_hashes.values() if len(flist) > 1]
total_dup_copies = sum(len(flist) - 1 for flist in dup_groups)
print(f"Identical Content Groups (Hash) : {len(dup_groups)} ({total_dup_copies} redundant copies)")

unique_records = []
seen_hashes = set()
for r in dataset_records:
    if r["hash"] not in seen_hashes:
        unique_records.append(r)
        seen_hashes.add(r["hash"])

print(f"Clean Unique Images Retained    : {len(unique_records)}")
print("=" * 65)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

class_to_records = defaultdict(list)
for r in unique_records:
    class_to_records[r["class_name"]].append(r)

classes = sorted(list(class_to_records.keys()))

for idx, c_name in enumerate(classes):
    sample = class_to_records[c_name][0]
    img = Image.open(sample["filepath"]).convert("RGB")
    axes[idx].imshow(img)
    short_label = c_name.replace("Tomato___", "").replace("_", " ")
    axes[idx].set_title(f"{short_label}\n({len(class_to_records[c_name])} imgs)", fontsize=11, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("🌿 PlantVillage Tomato Dataset — Class Inspection (10 Categories)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 2. Reproducible Stratified Dataset Splitting
To ensure scientifically valid results with **zero data leakage**:
- **Train Split (80%):** Model weights optimization with data augmentation.
- **Validation Split (10%):** Hyperparameter tuning and model checkpoint selection.
- **Test Split (10%):** Strictly held-out, untouched until final evaluation.
- **Stratification:** Maintains identical class distributions across all three splits.

In [ ]:
from sklearn.model_selection import train_test_split

df_data = pd.DataFrame(unique_records)
X = df_data["filepath"].values
y = df_data["class_name"].values

class_names = sorted(list(df_data["class_name"].unique()))
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for i, name in enumerate(class_names)}
y_indices = np.array([class_to_idx[name] for name in y])

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_indices
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("=" * 65)
print("📊 Stratified Dataset Split Summary (Seed 42)")
print("=" * 65)
print(f"Total Clean Images : {len(df_data)}")
print(f"Training Set (80%) : {len(X_train):>5} images")
print(f"Validation Set (10%): {len(X_val):>5} images")
print(f"Test Set (10%)     : {len(X_test):>5} images")
print("=" * 65)

train_set_files = set(X_train)
val_set_files = set(X_val)
test_set_files = set(X_test)
assert len(train_set_files.intersection(val_set_files)) == 0, "Train-Val Leakage Detected!"
assert len(train_set_files.intersection(test_set_files)) == 0, "Train-Test Leakage Detected!"
assert len(val_set_files.intersection(test_set_files)) == 0, "Val-Test Leakage Detected!"
print("✅ Verified: Absolute zero data leakage between Train, Val, and Test splits.")

In [ ]:
class PlantVillageDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        path = self.filepaths[idx]
        label = self.labels[idx]
        with Image.open(path) as img:
            img_rgb = img.convert("RGB")
        if self.transform is not None:
            img_tensor = self.transform(img_rgb)
        else:
            img_tensor = transforms.ToTensor()(img_rgb)
        return img_tensor, label

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE = (224, 224)

train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = PlantVillageDataset(X_train, y_train, transform=train_transforms)
val_dataset   = PlantVillageDataset(X_val, y_val, transform=eval_transforms)
test_dataset  = PlantVillageDataset(X_test, y_test, transform=eval_transforms)

BATCH_SIZE = 64 if torch.cuda.is_available() else 16
NUM_WORKERS = 4 if torch.cuda.is_available() else 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True if torch.cuda.is_available() else False)

print(f"DataLoaders initialized successfully.")
print(f"Batch Size: {BATCH_SIZE} | Workers: {NUM_WORKERS} | Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

## 3. Model Architecture — MobileNetV3-Large
We use **MobileNetV3-Large** with transfer learning from ImageNet.  
* **Why MobileNetV3-Large?** It achieves high accuracy on foliar diseases while remaining lightweight (~3.2M parameters, ~15 MB disk size).  
* **FastAPI Backend Friendly:** Fast forward pass on standard CPUs (<20ms latency), making it ideal for the KrishiOS AI production server.

In [ ]:
def build_mobilenet_v3_large(num_classes=10):
    try:
        weights = models.MobileNet_V3_Large_Weights.DEFAULT
        model = models.mobilenet_v3_large(weights=weights)
        print("✅ Successfully loaded pretrained ImageNet weights for MobileNetV3-Large.")
    except Exception as e:
        print(f"⚠️ Pretrained weights download failed: {e}")
        print("🔄 Initializing MobileNetV3-Large without pretrained weights (cold start)...")
        model = models.mobilenet_v3_large(weights=None)
        
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model

model = build_mobilenet_v3_large(num_classes=len(class_names))
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")
print(f"Classifier Output    : {model.classifier[3]}")

In [ ]:
MAX_EPOCHS = 15
INITIAL_LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)

OUTPUT_DIR = Path("/kaggle/working/krishios_model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "best_model.pth"

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "lr": []
}

best_val_loss = float("inf")
best_val_acc = 0.0
patience_counter = 0
best_epoch = 0

print("=" * 75)
print("🚀 Starting MobileNetV3-Large Training on PlantVillage Tomato Dataset")
print("=" * 75)

training_start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    epoch_start = time.time()
    
    # Training
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for inputs, labels in train_loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data).item()
        total_train += inputs.size(0)
        
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]
    
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = correct_train / total_train
    
    # Validation
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data).item()
            total_val += inputs.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = correct_val / total_val
    epoch_duration = time.time() - epoch_start
    
    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    history["lr"].append(current_lr)
    
    print(f"Epoch [{epoch:02d}/{MAX_EPOCHS:02d}] "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc*100:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc*100:.2f}% | "
          f"LR: {current_lr:.6f} | Time: {epoch_duration:.1f}s")
    
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_val_acc = epoch_val_acc
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), BEST_CHECKPOINT_PATH)
        print(f"  ⭐ Checkpoint saved! Best Val Loss: {best_val_loss:.4f} (Val Acc: {best_val_acc*100:.2f}%)")
    else:
        patience_counter += 1
        print(f"  ⚠️ No improvement for {patience_counter}/{PATIENCE} epochs.")
        if patience_counter >= PATIENCE:
            print(f"🛑 Early stopping triggered at epoch {epoch}. Restoring best checkpoint from epoch {best_epoch}.")
            break

total_training_time = time.time() - training_start_time
print("=" * 75)
print(f"🎉 Training Finished in {total_training_time/60:.2f} minutes!")
print(f"🏆 Best Validation Loss: {best_val_loss:.4f} | Best Val Acc: {best_val_acc*100:.2f}% (Epoch {best_epoch})")
print("=" * 75)

## 4. Evaluation on Held-Out Test Set
We now load the best saved checkpoint and evaluate strictly on the **10% Test Set** (never seen during training or validation).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
import seaborn as sns

model.load_state_dict(torch.load(BEST_CHECKPOINT_PATH, map_location=device))
model.eval()

all_preds = []
all_targets = []
all_probs = []

test_start = time.time()
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

test_duration = time.time() - test_start

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_probs = np.array(all_probs)

test_acc = np.mean(all_preds == all_targets)

top3_preds = np.argsort(all_probs, axis=1)[:, -3:]
top3_correct = [target in top3 for target, top3 in zip(all_targets, top3_preds)]
top3_acc = np.mean(top3_correct)

precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(all_targets, all_preds, average="macro", zero_division=0)

print("=" * 75)
print("📊 FINAL UNTOUCHED TEST SET EVALUATION REPORT")
print("=" * 75)
print(f"Test Samples Evaluated   : {len(all_targets):,}")
print(f"Top-1 Test Accuracy      : {test_acc*100:.2f}%")
print(f"Top-3 Test Accuracy      : {top3_acc*100:.2f}%")
print(f"Macro Precision          : {precision_macro*100:.2f}%")
print(f"Macro Recall             : {recall_macro*100:.2f}%")
print(f"Macro F1-Score           : {f1_macro*100:.2f}%")
print(f"Evaluation Latency       : {test_duration:.2f}s ({len(all_targets)/test_duration:.1f} samples/sec)")
print("=" * 75)

clean_class_names = [c.replace("Tomato___", "").replace("_", " ") for c in class_names]
print("\nPer-Class Classification Report:")
print(classification_report(all_targets, all_preds, target_names=clean_class_names, digits=4))

In [ ]:
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt="d", 
    cmap="Greens", 
    xticklabels=clean_class_names, 
    yticklabels=clean_class_names,
    cbar_kws={"label": "Sample Count"}
)
plt.title(f"Confusion Matrix — PlantVillage Tomato Classifier (Test Acc: {test_acc*100:.2f}%)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Class", fontsize=12, fontweight="bold")
plt.ylabel("Ground Truth Class", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

cm_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=300)
print(f"✅ Confusion matrix saved to: {cm_path}")
plt.show()

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], "o-", label="Train Loss", color="#10b981", linewidth=2)
plt.plot(range(1, len(history["val_loss"]) + 1), history["val_loss"], "s--", label="Val Loss", color="#ef4444", linewidth=2)
plt.axvline(best_epoch, color="#6366f1", linestyle=":", label=f"Best Checkpoint (Ep {best_epoch})")
plt.title("Cross-Entropy Loss Progression", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history["train_acc"]) + 1), [a * 100 for a in history["train_acc"]], "o-", label="Train Acc", color="#10b981", linewidth=2)
plt.plot(range(1, len(history["val_acc"]) + 1), [a * 100 for a in history["val_acc"]], "s--", label="Val Acc", color="#ef4444", linewidth=2)
plt.axvline(best_epoch, color="#6366f1", linestyle=":", label=f"Best Checkpoint (Ep {best_epoch})")
plt.title("Accuracy Progression (%)", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, alpha=0.3)

curves_path = OUTPUT_DIR / "training_curves.png"
plt.tight_layout()
plt.savefig(curves_path, dpi=300)
print(f"✅ Training curves saved to: {curves_path}")
plt.show()

## 5. Artifact Packaging & Standalone Inference Module
Export all artifacts into `/kaggle/working/krishios_model/` and bundle them into a downloadable ZIP archive (`krishios_model.zip`).

In [ ]:
# 1. Save class_mapping.json
class_mapping_data = {
    "idx_to_class": {str(k): v for k, v in idx_to_class.items()},
    "class_to_idx": {k: int(v) for k, v in class_to_idx.items()},
    "class_names": class_names,
    "clean_class_names": {c: c.replace("Tomato___", "").replace("_", " ") for c in class_names}
}
with open(OUTPUT_DIR / "class_mapping.json", "w") as f:
    json.dump(class_mapping_data, f, indent=2)

# 2. Save model_config.json
model_config_data = {
    "model_name": "MobileNetV3-Large",
    "crop_target": "Tomato",
    "dataset": "PlantVillage",
    "num_classes": len(class_names),
    "input_size": [224, 224],
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
    "train_val_test_split": [0.8, 0.1, 0.1],
    "random_seed": SEED,
    "framework": f"PyTorch {torch.__version__}",
    "torchvision_version": torchvision.__version__,
    "training_date": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
    "best_epoch": best_epoch,
    "early_stopping_patience": PATIENCE
}
with open(OUTPUT_DIR / "model_config.json", "w") as f:
    json.dump(model_config_data, f, indent=2)

# 3. Save training_metrics.json
metrics_data = {
    "test_accuracy": round(float(test_acc), 4),
    "test_top3_accuracy": round(float(top3_acc), 4),
    "macro_f1": round(float(f1_macro), 4),
    "macro_precision": round(float(precision_macro), 4),
    "macro_recall": round(float(recall_macro), 4),
    "best_validation_loss": round(float(best_val_loss), 4),
    "best_validation_accuracy": round(float(best_val_acc), 4),
    "total_training_epochs_executed": len(history["train_loss"]),
    "best_epoch": best_epoch
}
with open(OUTPUT_DIR / "training_metrics.json", "w") as f:
    json.dump(metrics_data, f, indent=2)

# 4. Save requirements.txt
requirements_content = """torch>=2.0.0
torchvision>=0.15.0
pillow>=9.0.0
numpy>=1.23.0
"""
with open(OUTPUT_DIR / "requirements.txt", "w") as f:
    f.write(requirements_content)

# 5. Save README.txt
readme_lines = [
    "KrishiOS AI — PlantVillage Tomato Disease Classification Model",
    "==================================================================",
    "Model Architecture: MobileNetV3-Large",
    "Target Crop       : Tomato (Solanum lycopersicum)",
    "Classes Count     : 10",
    f"Test Accuracy     : {test_acc*100:.2f}%",
    f"Top-3 Accuracy    : {top3_acc*100:.2f}%",
    f"Macro F1-Score    : {f1_macro*100:.2f}%",
    "",
    "Files in this archive:",
    "1. best_model.pth          - PyTorch state_dict for MobileNetV3-Large",
    "2. class_mapping.json      - Index to PlantVillage class name mappings",
    "3. model_config.json       - Input size, normalization constants, and metadata",
    "4. training_metrics.json   - Verified evaluation metrics on untouched test set",
    "5. confusion_matrix.png    - High-resolution confusion matrix heatmap",
    "6. training_curves.png     - Loss and accuracy progression graphs",
    "7. inference.py            - Standalone Python inference predictor module",
    "8. requirements.txt        - Minimal Python dependencies for inference",
    "9. README.txt              - Model documentation and specifications",
    "",
    "Integration Target:",
    "Extract all files into: D:/KrishiOS AI/backend/models/",
]
with open(OUTPUT_DIR / "README.txt", "w") as f:
    f.write("\n".join(readme_lines))

# 6. Save inference.py
inference_lines = [
    "import io",
    "import json",
    "from pathlib import Path",
    "from PIL import Image",
    "import torch",
    "import torch.nn as nn",
    "from torchvision import transforms, models",
    "",
    "class TomatoDiseasePredictor:",
    "    def __init__(self, model_dir=None, device=None):",
    "        if model_dir is None:",
    "            self.model_dir = Path(__file__).resolve().parent",
    "        else:",
    "            self.model_dir = Path(model_dir)",
    "        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')",
    "        mapping_path = self.model_dir / 'class_mapping.json'",
    "        with open(mapping_path, 'r') as f:",
    "            data = json.load(f)",
    "            self.idx_to_class = {int(k): v for k, v in data['idx_to_class'].items()}",
    "            self.clean_names = data.get('clean_class_names', {})",
    "        self.num_classes = len(self.idx_to_class)",
    "        self.model = models.mobilenet_v3_large(weights=None)",
    "        in_features = self.model.classifier[3].in_features",
    "        self.model.classifier[3] = nn.Linear(in_features, self.num_classes)",
    "        weights_path = self.model_dir / 'best_model.pth'",
    "        state = torch.load(weights_path, map_location=self.device)",
    "        self.model.load_state_dict(state)",
    "        self.model.to(self.device)",
    "        self.model.eval()",
    "        self.transform = transforms.Compose([",
    "            transforms.Resize((256, 256)),",
    "            transforms.CenterCrop((224, 224)),",
    "            transforms.ToTensor(),",
    "            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])",
    "        ])",
    "",
    "    def predict(self, image_input):",
    "        if isinstance(image_input, (str, Path)):",
    "            img = Image.open(image_input).convert('RGB')",
    "        elif isinstance(image_input, bytes):",
    "            img = Image.open(io.BytesIO(image_input)).convert('RGB')",
    "        elif isinstance(image_input, Image.Image):",
    "            img = image_input.convert('RGB')",
    "        else:",
    "            raise ValueError('Unsupported image input type.')",
    "        tensor = self.transform(img).unsqueeze(0).to(self.device)",
    "        with torch.no_grad():",
    "            logits = self.model(tensor)",
    "            probs = torch.softmax(logits, dim=1)[0].cpu().numpy()",
    "        top_idx = int(probs.argmax())",
    "        top_prob = float(probs[top_idx])",
    "        raw_class = self.idx_to_class[top_idx]",
    "        clean_name = self.clean_names.get(raw_class, raw_class)",
    "        top3_indices = probs.argsort()[-3:][::-1]",
    "        top3 = [",
    "            {",
    "                'class_name': self.idx_to_class[int(i)],",
    "                'display_name': self.clean_names.get(self.idx_to_class[int(i)], self.idx_to_class[int(i)]),",
    "                'confidence': round(float(probs[i]), 4)",
    "            }",
    "            for i in top3_indices",
    "        ]",
    "        return {",
    "            'predicted_class': raw_class,",
    "            'display_name': clean_name,",
    "            'confidence': round(top_prob, 4),",
    "            'top_3': top3",
    "        }",
]
with open(OUTPUT_DIR / "inference.py", "w") as f:
    f.write("\n".join(inference_lines))

print("=" * 65)
print("✅ All 9 artifacts successfully generated in:")
print(f"   {OUTPUT_DIR.resolve()}")
for item in sorted(OUTPUT_DIR.iterdir()):
    print(f"   • {item.name:<25} ({item.stat().st_size / 1024:.1f} KB)")
print("=" * 65)

In [ ]:
sys.path.insert(0, str(OUTPUT_DIR))
from inference import TomatoDiseasePredictor

predictor = TomatoDiseasePredictor(model_dir=OUTPUT_DIR, device=device)

sample_test_idx = 0
sample_test_file = X_test[sample_test_idx]
sample_true_class = class_names[y_test[sample_test_idx]]

result = predictor.predict(sample_test_file)

print("=" * 65)
print("🧪 Testing Standalone Inference Module:")
print("=" * 65)
print(f"Sample File      : {sample_test_file}")
print(f"Ground Truth     : {sample_true_class}")
print(f"Predicted Class  : {result['predicted_class']}")
print(f"Display Name     : {result['display_name']}")
print(f"Confidence Score : {result['confidence'] * 100:.2f}%")
print("\nTop-3 Probabilities:")
for rank, p in enumerate(result["top_3"], 1):
    print(f"  {rank}. {p['display_name']:<30} : {p['confidence']*100:.2f}%")
print("=" * 65)

In [ ]:
zip_base = Path("/kaggle/working/krishios_model")
zip_output = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

zip_size_mb = Path(zip_output).stat().st_size / (1024 * 1024)

print("=" * 65)
print("📦 KRISHIOS MODEL PACKAGE CREATED SUCCESSFULLY!")
print("=" * 65)
print(f"Zip Location : {zip_output}")
print(f"Zip Size     : {zip_size_mb:.2f} MB")
print("\n👉 HOW TO DOWNLOAD TO YOUR LOCAL KRISHIOS AI PROJECT:")
print("   1. Open the Kaggle right sidebar (or Output tab).")
print("   2. Locate '/kaggle/working/krishios_model.zip'.")
print("   3. Click the 3 dots (...) -> 'Download'.")
print("   4. Extract contents to: D:/KrishiOS AI/backend/models/")
print("=" * 65)

## 6. KrishiOS AI Backend Integration Guide

Once `krishios_model.zip` is downloaded and extracted into `D:\KrishiOS AI\backend\models\`:

1. **Verify Extracted Files:**
   * `D:\KrishiOS AI\backend\models\best_model.pth`
   * `D:\KrishiOS AI\backend\models\class_mapping.json`
   * `D:\KrishiOS AI\backend\models\model_config.json`
   * `D:\KrishiOS AI\backend\models\inference.py`

2. **Integration into `backend/app/services/disease_service.py`:**
   * The `DiseaseService` loads `best_model.pth` once on application startup.
   * Uploaded leaf images are fed through `TomatoDiseasePredictor` with exact ImageNet normalization (`224x224`).
   * The predicted category maps directly to `crop_type: "Tomato"`, `disease_name`, `confidence`.

3. **Separation of Concerns:**
   * **Disease Classification:** Validated by this trained model.
   * **Severity Detection & Progression:** Clearly tagged as experimental/heuristic in the API schema until dedicated segmentation models are trained.

---
**KrishiOS AI Engineering Team** — Clean, reproducible, and production-ready.